In [7]:
pip install ultralytics openai pillow requests # Установка зависимостей

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.6 MB/s  0:00:001.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openai]━━━━ 1/2 [openai]
Note: you may need to restart the kernel to use updated packages.


In [17]:
import cv2
from ultralytics import YOLO
from openai import OpenAI
from PIL import Image
import base64
import io
import os
import re

LM_STUDIO_URL = "http://127.0.0.1:1234/v1"
API_KEY = "lm-studio"
MODEL_NAME = "qwen/qwen3.5-9b"

NON_PRODUCT_CLASSES = {
    "person", "people", "man", "woman", "child",
    "face", "hand", "finger", "body",
}

det_model = YOLO("yolov8n.pt")
client = OpenAI(base_url=LM_STUDIO_URL, api_key=API_KEY)


def detect_and_crop(image_path: str):
    results = det_model(image_path, conf=0.4, iou=0.5)
    boxes = results[0].boxes

    if len(boxes) == 0:
        return "неизвестный объект", 0.0, None

    sorted_boxes = sorted(boxes, key=lambda b: float(b.conf[0]), reverse=True)

    chosen = None
    for box in sorted_boxes:
        cls_name = det_model.names[int(box.cls[0])]
        if cls_name.lower() not in NON_PRODUCT_CLASSES:
            chosen = box
            break

    if chosen is None:
        chosen = sorted_boxes[0]

    x1, y1, x2, y2 = map(int, chosen.xyxy[0])
    cls_id = int(chosen.cls[0])
    conf = float(chosen.conf[0])
    obj_name = det_model.names[cls_id]

    img = Image.open(image_path)
    w, h = img.size
    pad = 20
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)

    cropped = img.crop((x1, y1, x2, y2))
    return obj_name, conf, cropped


def strip_think_blocks(text: str) -> str:
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    match = re.search(r"[🔹•]", text)
    if match and match.start() > 0:
        text = text[match.start():]
    return text.strip()


def encode_image(image_pil: Image.Image) -> str:
    buffer = io.BytesIO()
    image_pil.save(buffer, format="JPEG", quality=85)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def generate_ozon_description(image_pil: Image.Image, obj_name: str) -> str: # ЗДЕСЬ НАСТРОЙКА ПРОМПТА
    prompt = """/no_think
Посмотри на изображение и создай карточку товара для Ozon на русском языке.

Формат ответа (строго, без отступлений):

🔹 Название товара: [короткое, продающее название]

🔹 Основные характеристики:
• [характеристика 1]
• [характеристика 2]
• [характеристика 3]
• [характеристика 4]
• [характеристика 5]

🔹 Описание: [3-4 предложения — преимущества, применение, особенности]

🔹 Комплектация: [что входит]

Только готовый текст. Никаких пояснений, рассуждений, предисловий."""

    img_b64 = encode_image(image_pil)

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "Ты копирайтер маркетплейса. Отвечай ТОЛЬКО на русском языке. Никаких рассуждений — сразу готовый результат.",
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
                        },
                    ],
                },
            ],
            temperature=0.4,
            max_tokens=2000, # Надо увеличить, чтобы больше описание давал, если не хватает.
            top_p=0.9,
            extra_body={"chat_template_kwargs": {"thinking": False}},
        )
        raw = response.choices[0].message.content
        return strip_think_blocks(raw)

    except Exception as e:
        return f"Ошибка при генерации описания: {e}"

if __name__ == "__main__":
    image_path = "Downloads/ziv6i08djvh11.jpg" # Сюда загружается фотография объекта. ОБЯЗАТЕЛЬНО КОНВЕРТИРОВАТЬ В jpg

    if not os.path.exists(image_path):
        print(f"Файл {image_path} не найден.")
        exit(1)

    print("Распознавание объекта...")
    obj_name, conf, cropped_img = detect_and_crop(image_path)
    print(f"Распознано: {obj_name} (уверенность: {conf:.2f})")

    if cropped_img is None:
        print("Объект не найден на изображении.")
        exit(1)

    print("Генерация описания через LM Studio...")
    description = generate_ozon_description(cropped_img, obj_name)

    print("\nОПИСАНИЕ ДЛЯ КАРТОЧКИ ТОВАРА:")
    print(description)

Распознавание объекта...

image 1/1 /Users/artemosipov/Downloads/ziv6i08djvh11.jpg: 640x480 1 person, 1 cell phone, 42.9ms
Speed: 6.8ms preprocess, 42.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 480)
Распознано: cell phone (уверенность: 0.46)
Генерация описания через LM Studio...

ОПИСАНИЕ ДЛЯ КАРТОЧКИ ТОВАРА:
🔹 Название товара: Pepsi Cola - Газированный напиток, классический вкус, 355 мл

🔹 Основные характеристики:
• Объем: 355 мл
• Бренд: Pepsi
• Тип напитка: Кола с газом
• Упаковка: Алюминиевая банка
• Калорийность: 150 ккал на 100 г

🔹 Описание: Легендарный газированный напиток с фирменным освежающим вкусом. Идеально подходит для утоления жажды в жаркий день или как дополнение к любимым блюдам. Насыщенный вкус и легкая пена гарантируют привычное удовольствие от каждого глотка.

🔹 Комплектация: 1 банка (355 мл)
